# GM Active Travel Equity — Reproducible Analysis

This notebook walks through the full analysis pipeline for the project:
_Does active travel infrastructure provision across Greater Manchester vary systematically with neighbourhood deprivation?_

Each section corresponds to a script in `src/`. You can run this notebook end-to-end after running `python src/ingest.py`, or step through it to understand how each stage works.

**Prerequisites**: raw data in `data/raw/` — run `src/ingest.py` first, or run the ingestion cell below.

In [ ]:
import sys
sys.path.insert(0, '..')   # make src/ importable when running from notebooks/

from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RAW_DIR  = Path('../data/raw')
PROC_DIR = Path('../data/processed')
OUT_DIR  = Path('../outputs')

PROC_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
(OUT_DIR / 'maps').mkdir(parents=True, exist_ok=True)

print('Environment ready.')

## 1. Data ingestion

All five sources download automatically — no manual steps needed.
The OSM cycling data is pulled via osmnx / Overpass API and requires no account.

In [ ]:
from src.ingest import (
    download_lsoa_boundaries,
    download_lad_boundaries,
    download_imd,
    download_census_travel,
    download_cycling_osm,
)

download_lsoa_boundaries(RAW_DIR)
download_lad_boundaries(RAW_DIR)
download_imd(RAW_DIR)
download_census_travel(RAW_DIR)
download_cycling_osm(RAW_DIR)

## 2. Cleaning and spatial processing

This section:
1. Loads LSOA boundaries and confirms they are filtered to the ten GM boroughs.
2. Clips OSM cycling features to each LSOA polygon and computes infrastructure density (m per km²).
3. Joins IMD 2019 deprivation scores via the ONS 2011-to-2021 LSOA crosswalk.
4. Computes active travel mode share from Census 2021 TS061.

In [ ]:
from src.clean import (
    load_gm_lsoas,
    load_cycling_osm,
    compute_infrastructure_density,
    load_imd,
    load_census_travel,
    build_analysis_dataset,
    add_borough_names,
)

gm_lsoas = load_gm_lsoas(RAW_DIR)
print(f'GM LSOAs loaded: {len(gm_lsoas)}')
gm_lsoas.head(3)

In [ ]:
cycling_gdf = load_cycling_osm(RAW_DIR, gm_lsoas)
print(f'OSM cycling features in GM: {len(cycling_gdf)}')
cycling_gdf.head(3)

In [ ]:
infra_df = compute_infrastructure_density(cycling_gdf, gm_lsoas)
infra_df.describe().round(1)

In [ ]:
imd_df    = load_imd(RAW_DIR)
census_df = load_census_travel(RAW_DIR)

analysis = build_analysis_dataset(gm_lsoas, infra_df, imd_df, census_df, raw_dir=RAW_DIR)
analysis = add_borough_names(analysis, RAW_DIR)

# Save to disk
analysis.to_file(PROC_DIR / 'gm_lsoa_analysis.gpkg', driver='GPKG')
analysis.drop(columns='geometry').to_csv(PROC_DIR / 'gm_lsoa_analysis.csv', index=False)

print(f'Analysis dataset shape: {analysis.shape}')
analysis.drop(columns='geometry').head()

## 3. Exploratory data analysis

Before modelling, I look at the distribution of key variables and their bivariate relationships.

In [ ]:
df = analysis.drop(columns='geometry')

print('=== Key variable summaries ===')
print(df[['infra_density_m_per_km2', 'imd_score', 'active_travel_pct',
           'cycling_pct', 'walking_pct']].describe().round(2))

In [ ]:
# Distribution of infrastructure density (raw vs log-transformed)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].hist(df['infra_density_m_per_km2'].dropna(), bins=60,
             color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].set_title('Infrastructure density (raw)')
axes[0].set_xlabel('m per km²')

axes[1].hist(np.log1p(df['infra_density_m_per_km2'].dropna()), bins=60,
             color='steelblue', edgecolor='white', linewidth=0.3)
axes[1].set_title('Infrastructure density (log₁₊ₓ transformed)')
axes[1].set_xlabel('log(1 + m per km²)')

for ax in axes:
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Proportion of LSOAs with zero infrastructure by quintile
zero_pct = (
    df.dropna(subset=['imd_quintile'])
    .assign(zero_infra=lambda x: x['infra_density_m_per_km2'] == 0)
    .groupby('imd_quintile')['zero_infra']
    .mean() * 100
).round(1)

print('% of LSOAs with zero cycling infrastructure by IMD quintile:')
print(zero_pct.rename(index={1:'Q1 (most deprived)',2:'Q2',3:'Q3',4:'Q4',5:'Q5 (least deprived)'}))

In [ ]:
from src.analyse import compute_correlations

corr_table = compute_correlations(df)
print(corr_table.to_string(index=False))

## 4. Statistical analysis

Two OLS regression models:

- **Baseline**: `active_travel_pct ~ imd_score + log_infra_density`
- **Full model**: adds borough fixed effects to control for unobserved borough-level factors (topography, bus network coverage, etc.)

Both use HC3 heteroskedasticity-robust standard errors.

In [ ]:
from src.analyse import (
    run_ols_regression,
    run_ols_no_fe,
    regression_diagnostics,
    describe_by_quintile,
    format_regression_summary,
)

quintile_stats = describe_by_quintile(df)
print(quintile_stats[[
    'imd_quintile_label', 'n',
    'infra_density_mean', 'infra_density_median',
    'active_travel_mean', 'active_travel_median',
]].to_string(index=False))

In [ ]:
results_fe   = run_ols_regression(df)
results_base = run_ols_no_fe(df)
diag         = regression_diagnostics(results_fe)

print(results_fe.summary())

In [ ]:
# Save regression summary
summary_text = format_regression_summary(results_fe, results_base, diag)
(OUT_DIR / 'regression_summary.txt').write_text(summary_text, encoding='utf-8')
print(summary_text)

## 5. Visualisations

Generates all four static figures and the interactive choropleth map.

In [ ]:
from src.visualise import (
    plot_infra_by_quintile,
    plot_deprivation_vs_mode_share,
    plot_density_by_borough,
    plot_cycling_vs_walking,
    build_interactive_map,
)

fig_dir = OUT_DIR / 'figures'
map_dir = OUT_DIR / 'maps'

plot_infra_by_quintile(df, fig_dir)
plot_deprivation_vs_mode_share(df, fig_dir)
plot_density_by_borough(df, fig_dir)
plot_cycling_vs_walking(df, fig_dir)

In [ ]:
# Display figures inline
from IPython.display import Image, display
for fig_path in sorted(fig_dir.glob('*.png')):
    display(Image(filename=str(fig_path)))

In [ ]:
# Build interactive map
map_path = build_interactive_map(analysis, map_dir)
print(f'Map saved to: {map_path}')

from IPython.display import IFrame
IFrame(src=str(map_path), width='100%', height=550)

## 6. Key findings

**Active travel is highest in the most deprived areas.** Q1 (most deprived) LSOAs have a mean active travel mode share of 12.5%, falling to 4.9% in Q5. The gradient is consistent across quintiles.

**Walking drives the pattern, not cycling.** Q1 LSOAs average 10.5% walking mode share against 3.8% in Q5. Spearman rho between IMD score and walking share is 0.71 (p < 0.001). The most likely explanation is constrained transport choices rather than good provision — lower car ownership and cost barriers to public transport make walking the default in the most deprived areas.

**Infrastructure density is marginally higher in deprived areas.** Pearson r = 0.09 (p < 0.001) between IMD score and infrastructure density. This reflects the inner-city geography of GM — Manchester and Salford are both relatively deprived and reasonably well-supplied with cycling routes. Q1 mean: 1,054 m/km², Q5 mean: 655 m/km².

**OLS regression confirms both deprivation and infrastructure matter.** Each additional IMD point is associated with a 0.16 percentage point increase in active travel share (95% CI: 0.15–0.17, p < 0.001). Log infrastructure density is also positively significant (coef = 0.09, p < 0.001). Borough fixed effects lift R² from 0.36 to 0.42; Manchester and Salford show notably higher active travel at equivalent deprivation levels.

## 7. Limitations

- IMD 2019 uses 2011 LSOA boundaries; the ONS 2011-to-2021 crosswalk is applied, but substantially redrawn LSOAs will have some deprivation score mismatch.
- OSM cycling infrastructure coverage depends on volunteer contributor activity. Urban core areas in GM are generally well-mapped, but infrastructure quality (surface condition, physical separation) cannot be captured from OSM tags.
- Census 2021 travel-to-work captures usual method, not actual trips, and excludes non-workers. Areas with high unemployment may appear to have lower active travel shares simply because fewer residents are counted.
- OLS assumes linearity; a small number of high-density LSOAs could be influential. The log transformation reduces but does not eliminate this.